
# Experiment 3 - Fine-Tuned Aya Expanse 8B + Retrieval-Augmented Generation
### Multilingual Health Question Answering in Low-Resource African Languages

**Goal of this notebook:** combine the LoRA-fine-tuned model from Notebook 2 with the language-aware FAISS
retrieval indexes from Notebook 1. Crucially, we follow the **training/inference alignment principle** from
the project brief (Section 12): the model is *re-fine-tuned* here with retrieved examples already present
in its training prompts, so it learns to actually use the retrieved context rather than ignoring it. Simply
bolting RAG onto the Notebook 2 adapter at inference time (without ever training on RAG-style prompts) would
violate that alignment principle, so this notebook trains a **second, RAG-aware LoRA adapter** starting
from the same base model.

This notebook reuses, without recomputation:
- the **FAISS indexes + embedding metadata** saved by Notebook 1 (`exp1_rag/`)
- the same **base model** (`CohereForAI/aya-expanse-8b`) used in Notebook 2, loaded fresh in 4-bit

It does **not** reuse the Notebook 2 adapter directly as a starting point for further training, since LoRA
adapters are tied to the exact prompt distribution they were trained on; instead it trains a clean adapter
on RAG-formatted prompts so the comparison between Experiment 2 and Experiment 3 isolates the effect of
adding retrieval, not an artifact of sequential fine-tuning.

**Outputs saved by this notebook:**
- `lora_adapter_rag/` - the RAG-aware LoRA adapter
- `training_logs.json` - loss curves
- `submission_exp3_finetuned_plus_rag.csv`
- `exp3_validation_metrics.json`
- final cross-experiment comparison plot covering Experiments 1, 2, and 3



## 0. Environment Setup


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR  = '/content/drive/MyDrive/health_qa_project'
DATA_DIR     = f'{PROJECT_DIR}/data'
ARTIFACT_DIR = f'{PROJECT_DIR}/artifacts'
SUB_DIR      = f'{PROJECT_DIR}/submissions'

EXP1_DIR     = f'{ARTIFACT_DIR}/exp1_rag'        # FAISS indexes live here (from Notebook 1)
EXP2_DIR     = f'{ARTIFACT_DIR}/exp2_finetune'   # reference only, for comparison plots
EXP3_DIR     = f'{ARTIFACT_DIR}/exp3_finetune_rag'
ADAPTER_DIR  = f'{EXP3_DIR}/lora_adapter_rag'

for d in [EXP3_DIR, ADAPTER_DIR]:
    os.makedirs(d, exist_ok=True)

# Sanity check: confirm Notebook 1's artifacts exist before we go further
assert os.path.exists(f'{EXP1_DIR}/manifest.json'), (
    f'Could not find {EXP1_DIR}/manifest.json - run Notebook 1 (Pure RAG) first, '
    'it builds and saves the FAISS indexes this notebook depends on.'
)
print('Found Experiment 1 FAISS artifacts at:', EXP1_DIR)
print('Experiment 3 outputs will be saved under:', EXP3_DIR)


In [ ]:

!pip install -q -U bitsandbytes peft transformers accelerate trl rouge-score sentencepiece datasets sentence-transformers faiss-cpu
print('Dependencies installed')


In [ ]:

import re
import json
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from rouge_score import rouge_scorer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))



## 1. Configuration

Most settings mirror Notebook 2 so the two fine-tuning runs are comparable; the key addition is the
retrieval config (`TOP_K`, embedding model name) which must match Notebook 1 exactly since we are loading
its FAISS indexes directly rather than rebuilding them.


In [ ]:

# Paths
TRAIN_PATH      = f'{DATA_DIR}/train.csv'
VAL_PATH        = f'{DATA_DIR}/val.csv'
TEST_PATH       = f'{DATA_DIR}/test.csv'
SAMPLE_SUB_PATH = f'{DATA_DIR}/sample_submission.csv'

# Columns
ID_COL       = 'ID'
QUESTION_COL = 'input'
ANSWER_COL   = 'output'
LANG_COL     = 'subset'

# Model (must match Notebook 2's base model)
BASE_MODEL_NAME = 'CohereForAI/aya-expanse-8b'

# QLoRA config - identical to Notebook 2 so the only varying factor is the RAG context
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# Training config - sequence length increased vs Notebook 2 to accommodate retrieved
# reference examples in the prompt; everything else kept the same for a fair comparison.
MAX_SEQ_LENGTH        = 1024
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS      = 16
NUM_EPOCHS            = 3
LEARNING_RATE         = 2e-4
WARMUP_RATIO          = 0.03
LOGGING_STEPS         = 25
SAVE_STEPS            = 250
EVAL_STEPS            = 250

# Retrieval config - MUST match Notebook 1's manifest
TOP_K            = 3   # fewer than Notebook 1's default (5) to leave room for the longer base prompt
MAX_NEW_TOKENS   = 256

SUBSET_TO_LANGUAGE = {
    'Eng': 'English', 'Aka': 'Akan', 'Lug': 'Luganda', 'Swa': 'Swahili', 'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    return SUBSET_TO_LANGUAGE.get(subset_code.split('_')[0], subset_code)

with open(f'{EXP1_DIR}/manifest.json') as f:
    exp1_manifest = json.load(f)

EMBED_MODEL_NAME = exp1_manifest['embed_model_name']
print('Loaded retrieval config from Notebook 1 manifest:')
print(json.dumps(exp1_manifest, indent=2))
print()
notebook1_top_k = exp1_manifest['top_k_used_here']
print(f'Using TOP_K = {TOP_K} for this notebook (Notebook 1 used {notebook1_top_k}).')



## 2. Load and Prepare Data


In [ ]:

train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

def clean_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL] = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]   = train[ANSWER_COL].map(clean_text)
val[QUESTION_COL]   = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]     = val[ANSWER_COL].map(clean_text)
test[QUESTION_COL]  = test[QUESTION_COL].map(clean_text)

train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[QUESTION_COL] != ''].reset_index(drop=True)

train['language'] = train[LANG_COL].map(subset_to_language_name)
val['language']    = val[LANG_COL].map(subset_to_language_name)
test['language']   = test[LANG_COL].map(subset_to_language_name)

print(f'Train shape: {train.shape}, Val shape: {val.shape}, Test shape: {test.shape}')



## 3. Reload FAISS Retrieval Indexes from Notebook 1 (No Recomputation)

This is the key reuse step requested for this project: the embedding model, FAISS indexes, and metadata
were already built and saved in Notebook 1. We load them here directly rather than re-embedding 29k+
training questions a second time.


In [ ]:

print(f'Loading embedding model: {EMBED_MODEL_NAME} ...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print('Embedding model loaded.')


In [ ]:

language_indexes = {}

for lang in exp1_manifest['languages']:
    safe_lang = lang.replace(' ', '_')
    index_path = f'{EXP1_DIR}/faiss_{safe_lang}.index'
    meta_path  = f'{EXP1_DIR}/meta_{safe_lang}.pkl'

    assert os.path.exists(index_path) and os.path.exists(meta_path), (
        f'Missing FAISS artifacts for language {lang}. Re-run Notebook 1 to regenerate them.'
    )

    index = faiss.read_index(index_path)
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)

    language_indexes[lang] = {
        'faiss_index': index,
        'ids':         meta['ids'],
        'questions':   meta['questions'],
        'answers':     meta['answers'],
    }

print('Reloaded FAISS indexes for languages:', {k: v['faiss_index'].ntotal for k, v in language_indexes.items()})


In [ ]:

def retrieve_top_k(question: str, language: str, k: int = TOP_K):
    '''
    Retrieve the top-k most similar training examples for a question, restricted
    to the same language. Identical logic to Notebook 1, operating on the reloaded
    FAISS indexes rather than freshly built ones.
    '''
    if language not in language_indexes:
        return []

    payload = language_indexes[language]
    query_vec = embed_model.encode(
        ['query: ' + question],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype('float32')

    k = min(k, payload['faiss_index'].ntotal)
    scores, idxs = payload['faiss_index'].search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], idxs[0]):
        results.append({
            'question': payload['questions'][idx],
            'answer':   payload['answers'][idx],
            'score':    float(score),
        })
    return results

# Sanity check
sample_row = val.iloc[0]
sample_results = retrieve_top_k(sample_row[QUESTION_COL], sample_row['language'], k=TOP_K)
print(f'Retrieved {len(sample_results)} examples for a sample {sample_row["language"]} question.')



## 4. Build RAG-Aware Training Prompts (Training/Inference Alignment)

For every training example, we retrieve its own top-K nearest neighbours **excluding itself** and build
the same RAG prompt structure used at inference time (Section 11 of the brief). This is what makes the
fine-tuned model actually learn to read and use the retrieved context, rather than just seeing it for the
first time at test time.

**Important leakage guard:** since the retrieval corpus IS the training set, a training question retrieving
itself as a "reference example" would let the model trivially copy the answer instead of learning to
synthesize from context. We explicitly filter out the exact-ID self-match.


In [ ]:

def build_rag_prompt(question: str, language: str, retrieved_examples: list) -> str:
    lines = [
        'You are a multilingual health assistant.',
        f'Answer only in {language}.',
        '',
    ]
    for i, ex in enumerate(retrieved_examples, start=1):
        lines.append(f'Reference Example {i}')
        lines.append('Question:')
        lines.append(ex['question'])
        lines.append('Answer:')
        lines.append(ex['answer'])
        lines.append('')
    lines.append('Current Question:')
    lines.append(question)
    lines.append('')
    lines.append('Answer:')
    return '\n'.join(lines)


def retrieve_top_k_excluding_self(question: str, language: str, self_id: str, k: int = TOP_K):
    '''Same as retrieve_top_k, but drops a retrieved example whose ID matches self_id
    (prevents a training row from "retrieving itself" and trivially copying its own answer).'''
    if language not in language_indexes:
        return []
    payload = language_indexes[language]

    # Retrieve a few extra candidates so we still have k after dropping a self-match
    raw_results = retrieve_top_k(question, language, k=k + 1)
    filtered = [r for r in raw_results if r['question'] != question][:k]
    return filtered


print('Building RAG-formatted training prompts (this re-uses the FAISS indexes, no re-embedding of the corpus)...')

train_rag_prompts = []
for _, row in train.iterrows():
    retrieved = retrieve_top_k_excluding_self(row[QUESTION_COL], row['language'], row[ID_COL], k=TOP_K)
    train_rag_prompts.append(build_rag_prompt(row[QUESTION_COL], row['language'], retrieved))

train['rag_prompt'] = train_rag_prompts
train['training_text'] = train['rag_prompt'] + ' ' + train[ANSWER_COL]

print('Example RAG-formatted training text:')
print('=' * 70)
print(train['training_text'].iloc[0][:1200])
print('=' * 70)


In [ ]:

# Same for validation, used during training for eval_loss tracking
val_rag_prompts = []
for _, row in val.iterrows():
    retrieved = retrieve_top_k_excluding_self(row[QUESTION_COL], row['language'], row[ID_COL], k=TOP_K)
    val_rag_prompts.append(build_rag_prompt(row[QUESTION_COL], row['language'], retrieved))

val['rag_prompt'] = val_rag_prompts
val['training_text'] = val['rag_prompt'] + ' ' + val[ANSWER_COL]

print('RAG prompts built for train and validation sets.')


In [ ]:

# Token length check with the new RAG-formatted prompts - this is why MAX_SEQ_LENGTH was raised to 1024
tmp_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
sample_lengths = [len(tmp_tokenizer.encode(t)) for t in train['training_text'].sample(min(2000, len(train)), random_state=SEED)]

plt.figure(figsize=(8, 5))
plt.hist(sample_lengths, bins=50, color='#4C72B0')
plt.axvline(MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}')
plt.title('Token length distribution - RAG-formatted training examples')
plt.xlabel('Token count')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig(f'{EXP3_DIR}/token_length_distribution_rag.png', dpi=150)
plt.show()

pct_truncated = np.mean(np.array(sample_lengths) > MAX_SEQ_LENGTH) * 100
print(f'Percent of sampled examples exceeding MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH}): {pct_truncated:.2f}%')
del tmp_tokenizer



## 5. Load Base Model in 4-bit and Attach a Fresh LoRA Adapter


In [ ]:

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {BASE_MODEL_NAME} in 4-bit NF4 ...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



## 6. Fine-Tune the RAG-Aware Adapter


In [ ]:

train_dataset = Dataset.from_pandas(train[['training_text']].rename(columns={'training_text': 'text'}))
eval_dataset  = Dataset.from_pandas(val[['training_text']].rename(columns={'training_text': 'text'}))

sft_config = SFTConfig(
    output_dir=f'{EXP3_DIR}/checkpoints',
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy='steps',
    save_strategy='steps',
    save_total_limit=2,
    bf16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LENGTH,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    dataset_text_field='text',
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print('Trainer ready. Starting RAG-aware fine-tuning ...')


In [ ]:

train_start = time.time()
train_result = trainer.train()
train_minutes = (time.time() - train_start) / 60
print(f'Training complete in {train_minutes:.1f} minutes')


In [ ]:

log_history = trainer.state.log_history
with open(f'{EXP3_DIR}/training_logs.json', 'w') as f:
    json.dump(log_history, f, indent=2)

train_steps, train_losses = [], []
eval_steps_list, eval_losses = [], []

for entry in log_history:
    if 'loss' in entry and 'eval_loss' not in entry:
        train_steps.append(entry['step'])
        train_losses.append(entry['loss'])
    if 'eval_loss' in entry:
        eval_steps_list.append(entry['step'])
        eval_losses.append(entry['eval_loss'])

plt.figure(figsize=(9, 5))
plt.plot(train_steps, train_losses, label='Training loss', color='#4C72B0')
plt.plot(eval_steps_list, eval_losses, label='Validation loss', color='#DD8452', marker='o')
plt.xlabel('Training step')
plt.ylabel('Loss')
plt.title(f'Experiment 3: RAG-aware fine-tuning loss curve (Aya Expanse 8B, LoRA r={LORA_R}, top_k={TOP_K})')
plt.legend()
plt.tight_layout()
plt.savefig(f'{EXP3_DIR}/exp3_loss_curve.png', dpi=150)
plt.show()


In [ ]:

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_manifest = {
    'base_model': BASE_MODEL_NAME,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'lora_target_modules': LORA_TARGET_MODULES,
    'max_seq_length': MAX_SEQ_LENGTH,
    'top_k_retrieval': TOP_K,
    'embed_model': EMBED_MODEL_NAME,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'training_minutes': train_minutes,
}
with open(f'{EXP3_DIR}/adapter_manifest.json', 'w') as f:
    json.dump(adapter_manifest, f, indent=2)

print('RAG-aware LoRA adapter saved to:', ADAPTER_DIR)



## 7. Inference: Fine-Tuned Model + RAG

The full pipeline from the project brief (Section 13): extract language -> retrieve top-K examples ->
construct RAG prompt -> generate with the RAG-aware fine-tuned model.


In [ ]:

model.eval()
model.config.use_cache = True

def generate_rag_answers_batch(df, question_col=QUESTION_COL, lang_col='language', id_col=ID_COL,
                                 batch_size=4, max_new_tokens=MAX_NEW_TOKENS, exclude_self=True):
    answers = []
    prompts = []

    for _, row in df.iterrows():
        if exclude_self:
            retrieved = retrieve_top_k_excluding_self(row[question_col], row[lang_col], row[id_col], k=TOP_K)
        else:
            retrieved = retrieve_top_k(row[question_col], row[lang_col], k=TOP_K)
        prompts.append(build_rag_prompt(row[question_col], row[lang_col], retrieved))

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(
            batch_prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_SEQ_LENGTH
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
            )

        for i in range(len(batch_prompts)):
            full_text   = tokenizer.decode(output_ids[i], skip_special_tokens=True)
            prompt_text = tokenizer.decode(inputs['input_ids'][i], skip_special_tokens=True)
            answer = full_text[len(prompt_text):].strip()
            answers.append(answer)

        if start % (batch_size * 10) == 0:
            print(f'  Generated {start + len(batch_prompts)} / {len(prompts)}')

    return answers

print('Quick sanity check on one validation example ...')
sample_answer = generate_rag_answers_batch(val.iloc[[0]], batch_size=1)
print('Sample answer:', sample_answer[0][:300])


In [ ]:

class WhitespaceTokenizer:
    '''Whitespace tokeniser - language-agnostic, safe for African scripts.'''
    def tokenize(self, text):
        if text is None:
            return []
        return str(text).strip().split()

_scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False)

def compute_rouge(predictions, references):
    r1_scores, rl_scores = [], []
    for pred, ref in zip(predictions, references):
        score = _scorer.score(str(ref), str(pred))
        r1_scores.append(score['rouge1'].fmeasure)
        rl_scores.append(score['rougeL'].fmeasure)
    return {
        'rouge1_f1': float(np.mean(r1_scores)) if r1_scores else 0.0,
        'rougeL_f1': float(np.mean(rl_scores)) if rl_scores else 0.0,
    }

def compute_rouge_by_language(predictions, references, languages):
    results = {}
    lang_arr = np.array(languages)
    for lang in np.unique(lang_arr):
        mask    = lang_arr == lang
        preds_l = [p for p, m in zip(predictions, mask) if m]
        refs_l  = [r for r, m in zip(references,  mask) if m]
        results[lang] = compute_rouge(preds_l, refs_l)
    return pd.DataFrame(results).T

print('Generating predictions for the FULL validation set (fine-tuned model + RAG) ...')
val_preds_rag_finetuned = generate_rag_answers_batch(val)

metrics_rag_finetuned_val = compute_rouge(val_preds_rag_finetuned, val[ANSWER_COL].tolist())
print('Fine-tuned + RAG validation ROUGE:', metrics_rag_finetuned_val)


In [ ]:

rouge_by_lang_rag_finetuned = compute_rouge_by_language(
    val_preds_rag_finetuned, val[ANSWER_COL].tolist(), val['language'].tolist()
)
display(rouge_by_lang_rag_finetuned.round(4))

rouge_by_lang_rag_finetuned[['rouge1_f1', 'rougeL_f1']].plot(kind='bar', figsize=(10, 5), color=['#8172B2', '#937860'])
plt.title('Experiment 3: Fine-tuned Aya Expanse 8B + RAG - ROUGE by Language')
plt.ylabel('F1 score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{EXP3_DIR}/exp3_rouge_by_language.png', dpi=150)
plt.show()



## 8. Full Cross-Experiment Comparison (1 vs 2 vs 3)

This is the headline comparison table/plot for the final report: it pulls saved metrics from all three
notebooks, so re-running any one of them and re-running this cell will always reflect the latest numbers.


In [ ]:

exp1_metrics_path = f'{EXP1_DIR}/exp1_validation_metrics.json'
exp2_metrics_path = f'{EXP2_DIR}/exp2_validation_metrics.json'

rows = []

if os.path.exists(exp1_metrics_path):
    with open(exp1_metrics_path) as f:
        exp1_results = json.load(f)
    rows.append(('Exp 1: Pure RAG', exp1_results['full_validation_metrics']))
else:
    print('Warning: Experiment 1 metrics not found, skipping from comparison.')

if os.path.exists(exp2_metrics_path):
    with open(exp2_metrics_path) as f:
        exp2_results = json.load(f)
    rows.append(('Exp 2: Fine-tuned only', exp2_results['full_validation_metrics']))
else:
    print('Warning: Experiment 2 metrics not found, skipping from comparison.')

rows.append(('Exp 3: Fine-tuned + RAG', metrics_rag_finetuned_val))

comparison_all = pd.DataFrame({
    'Experiment': [r[0] for r in rows],
    'ROUGE-1 F1': [r[1]['rouge1_f1'] for r in rows],
    'ROUGE-L F1': [r[1]['rougeL_f1'] for r in rows],
})
display(comparison_all.round(4))

comparison_all.set_index('Experiment')[['ROUGE-1 F1', 'ROUGE-L F1']].plot(kind='bar', figsize=(9, 5), color=['#4C72B0', '#DD8452'])
plt.title('Cross-Experiment Comparison: Validation ROUGE Scores')
plt.ylabel('F1 score')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(f'{EXP3_DIR}/cross_experiment_comparison.png', dpi=150)
plt.show()

comparison_all.to_csv(f'{EXP3_DIR}/cross_experiment_comparison.csv', index=False)
print('Saved comparison table to:', f'{EXP3_DIR}/cross_experiment_comparison.csv')



## 9. Generate Test Predictions and Build Submission File


In [ ]:

def make_submission(ids, predictions, output_path):
    clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]

    sub = pd.DataFrame()
    sub['ID']         = ids
    sub['TargetRLF1'] = clean_preds
    sub['TargetR1F1'] = clean_preds
    sub['TargetLLM']  = clean_preds
    sub = sub[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]

    required_cols = ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert list(sub.columns) == required_cols
    assert len(sub) == len(test), f'Row count mismatch: {len(sub)} vs {len(test)}'
    assert sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].notna().all().all()
    assert (sub['TargetRLF1'] == sub['TargetR1F1']).all()
    assert (sub['TargetRLF1'] == sub['TargetLLM']).all()

    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f'Submission saved to: {output_path}')
    print(f'  Shape: {sub.shape}')
    display(sub.head(3))
    return sub

print('Generating test predictions with the fine-tuned + RAG pipeline ...')
# Test set has no ground-truth answers in the training corpus, so no self-exclusion is needed,
# but we keep the same function signature for consistency. Test rows simply won't match
# any training ID, so exclude_self has no effect here - left on for code-path consistency.
test_preds_exp3 = generate_rag_answers_batch(test, exclude_self=True)

sub_exp3 = make_submission(
    test[ID_COL].values, test_preds_exp3,
    f'{SUB_DIR}/submission_exp3_finetuned_plus_rag.csv'
)


In [ ]:

exp3_results = {
    'experiment': 'Experiment 3: Fine-tuned Aya Expanse 8B + RAG',
    'base_model': BASE_MODEL_NAME,
    'embed_model': EMBED_MODEL_NAME,
    'top_k_retrieval': TOP_K,
    'lora_config': {
        'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
        'target_modules': LORA_TARGET_MODULES,
    },
    'training_config': {
        'max_seq_length': MAX_SEQ_LENGTH,
        'effective_batch_size': PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'training_minutes': train_minutes,
    },
    'full_validation_metrics': metrics_rag_finetuned_val,
    'rouge_by_language': rouge_by_lang_rag_finetuned.round(4).to_dict(orient='index'),
}

with open(f'{EXP3_DIR}/exp3_validation_metrics.json', 'w') as f:
    json.dump(exp3_results, f, indent=2)

print('Saved Experiment 3 metrics to:', f'{EXP3_DIR}/exp3_validation_metrics.json')
print()
print('=== EXPERIMENT 3 SUMMARY ===')
print(json.dumps({k: v for k, v in exp3_results.items() if k != 'rouge_by_language'}, indent=2))
print()
print('=== FINAL CROSS-EXPERIMENT COMPARISON ===')
display(comparison_all.round(4))



## 10. Next Steps

- **Leaderboard checkpoint:** submit `submission_exp3_finetuned_plus_rag.csv` on Zindi and screenshot the
  score - this is the final, headline result for the report.
- All artifacts needed for the report (loss curves, ROUGE-by-language tables, the cross-experiment
  comparison plot and CSV) are saved under `EXP3_DIR` on Drive, alongside the equivalent files in
  `EXP1_DIR` and `EXP2_DIR` from the earlier notebooks.
- For the "at least 10 meaningful experiments" requirement: variations worth running and logging from this
  same notebook skeleton include changing `TOP_K` (e.g. 1, 3, 5), changing `LORA_R` (e.g. 8, 16, 32),
  toggling `exclude_self` during training data construction to quantify the leakage effect, and trying a
  different embedding model (e.g. `sentence-transformers/LaBSE`) for retrieval.
- This is the final experiment notebook. The academic report should now be written, citing the saved
  JSON metrics files, the loss-curve PNGs, and the leaderboard screenshots from all three submissions.
